# 07: Parameters and equations

Parameters are named values inside a part. Features can reference them by name, so changing one parameter propagates through every dimension that uses it.

## What you'll do
1. Build a small extrusion whose depth is a named parameter
2. List existing parameters
3. Drive the depth by changing the parameter
4. Add a free-standing parameter `A`, then a parameter `B = A * 2`
5. Push `A = 10`, watch `B` follow

**Prereq:** open a fresh empty part in Alibre.

## Setup: a parametric box

In [1]:
from alibrex import (
    CurrentPart,
    ADDirectionType,
    ADParameterType,
    ADPartFeatureEndCondition,
)

part = CurrentPart()

xy = part.DesignPlanes.Item(0)
sk = part.Sketches.AddSketch(None, xy, "Base")
figs = sk.Figures
figs.AddLine(0.0, 0.0, 4.0, 0.0)
figs.AddLine(4.0, 0.0, 4.0, 2.0)
figs.AddLine(4.0, 2.0, 0.0, 2.0)
figs.AddLine(0.0, 2.0, 0.0, 0.0)

part.Features.AddExtrudedBoss(
    sk, 1.0, ADPartFeatureEndCondition.AD_TO_DEPTH,
    None, None, 0.0,
    ADDirectionType.AD_ALONG_NORMAL, None, None, False,
    None, False,
    "Box", "BoxDepth", "",     # <-- the depth becomes a parameter named "BoxDepth"
)

<ComProxy(IADExtrusionFeature)>

## List every parameter on the part

In [2]:
params = part.Parameters
for i in range(params.Count):
    p = params.Item(i)
    eq = f"  =  {p.Equation}" if p.Equation else ""
    print(f"  {p.Name:20s} = {p.Value:8.4f}{eq}")

  thickness            =   0.2540
  D3                   =   0.2540  =  thickness
  D5                   =   0.7620
  D7                   =   0.7620
  D8                   =  -0.5080
  D2                   =  11.2279
  C2                   =   1.0000
  D6                   =   0.7620
  BoxDepth             =   1.0000
  D11                  =   0.0000
  D10                  =  -9.6520
  D1                   =  12.9432
  C1                   =   2.0000


## Find one by name

No direct lookup exists, so iterate.

In [3]:
depth = next(
    params.Item(i)
    for i in range(params.Count)
    if params.Item(i).Name == "BoxDepth"
)
depth.Value

1.0

## Change `BoxDepth` from 1.0 to 2.5 cm

Parameter mutations run inside a transaction so the part regenerates
consistently.

In [4]:
params.OpenParameterTransaction()
depth.Value = 2.5
params.CloseParameterTransaction()
part.RegenerateAll()
depth.Value

2.5

## Add a free-standing parameter `A`

In [5]:
a = params.NewParameter("A", ADParameterType.AD_DISTANCE)
params.OpenParameterTransaction()
a.Value = 3.0
params.CloseParameterTransaction()
a.Value

3.0

## Add parameter `B` driven by an equation `A * 2`

In [6]:
b = params.NewParameter("B", ADParameterType.AD_DISTANCE)
params.OpenParameterTransaction()
b.Equation = "A * 2"
params.CloseParameterTransaction()
part.RegenerateAll()
b.Value

6.0

## Push `A` to 10, `B` follows

In [7]:
params.OpenParameterTransaction()
a.Value = 10.0
params.CloseParameterTransaction()
part.RegenerateAll()
a.Value, b.Value

(10.0, 20.0)

## Final dump

In [8]:
for i in range(params.Count):
    p = params.Item(i)
    eq = f"  =  {p.Equation}" if p.Equation else ""
    print(f"  {p.Name:20s} = {p.Value:8.4f}{eq}")

  thickness            =   0.2540
  D3                   =   0.2540  =  thickness
  D5                   =   0.7620
  D7                   =   0.7620
  D8                   =  -0.5080
  D2                   =  11.2279
  C2                   =   1.0000
  D6                   =   0.7620
  BoxDepth             =   2.5000
  D11                  =   0.0000
  D10                  =  -9.6520
  D1                   =  12.9432
  C1                   =   2.0000
